In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from torchvision.transforms import functional as TF
import numpy as np
import pandas as pd
from PIL import Image
import os
import random
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc
from sklearn.preprocessing import label_binarize
from tqdm import tqdm
import warnings
import cv2
from collections import Counter
import time
import json

warnings.filterwarnings('ignore')

# Check CUDA availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version: {torch.version.cuda}")

# Set random seeds for reproducibility
def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)


In [ ]:

'''
Download these datasets, and extract them to the parent directory
https://www.kaggle.com/datasets/mayooo207/galaxy10augment/data
https://www.kaggle.com/datasets/mayooo207/galaxy10/data
'''

In [ ]:
class Config:
    """Centralized configuration for hyperparameters"""
    
    # Data parameters
    DATA_DIR = "../galaxy10Aug"
    IMG_SIZE = 256  # Paper uses 256x256 images
    NUM_CHANNELS = 3
    
    # Model architecture (from Table 3 in paper)
    INITIAL_CHANNELS = 64
    LAYER_CHANNELS = [128, 256, 512]  # Channels for Layer1, Layer2, Layer3
    NUM_DMA_BLOCKS = [2, 2, 2]  # Number of DMA blocks per layer
    
    # Training parameters (from Section 3.3)
    BATCH_SIZE = 32  # Paper: optimal batch size = 32
    LEARNING_RATE = 0.0001  # Paper: optimal lr = 0.0001
    NUM_EPOCHS = 150  # Paper: trained for 100 epochs
    WEIGHT_DECAY = 1e-4  # Standard for vision models
    
    # Data split ratios (from paper: 8:1:1)
    TRAIN_RATIO = 0.8
    VAL_RATIO = 0.1
    TEST_RATIO = 0.1
    
    # Augmentation parameters
    AUGMENTATION_PROB = 0.5  # Paper: 50% probability for flipping
    ROTATION_DEGREES = 30  # Paper: 30 degrees rotation for underrepresented classes
    
    # Preprocessing parameters (adding based on galaxy image characteristics) ( already applies on augmented data for training)
    APPLY_DENOISING = False  # Bilateral filter to reduce noise while preserving edges
    APPLY_CLAHE = False  # Adaptive histogram equalization for contrast enhancement
    GAMMA_CORRECTION = 1 # Slight gamma correction for faint galaxies
    
    # Training optimization
    USE_MIXED_PRECISION = True  # Enable for faster training
    GRADIENT_CLIP_VALUE = 1.0  # Gradient clipping for stability
    
    # Checkpointing
    CHECKPOINT_DIR = "checkpoints"
    SAVE_EVERY_N_EPOCHS = 10
    
    # Scheduler
    T_MAX = 100  # For cosine annealing (same as NUM_EPOCHS)
    MIN_LR = 1e-6
    
    # Early stopping
    PATIENCE = 15
    MIN_DELTA = 0.001

config = Config()


In [ ]:
class GalaxyDataset(Dataset):
    """
    Custom dataset for galaxy images with preprocessing and augmentation.
    Includes noise reduction and contrast enhancement specifically for astronomical images.
    """
    
    def __init__(self, image_paths, labels, transform=None, augment=False, config=config):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform
        self.augment = augment
        self.config = config
        
    def __len__(self):
        return len(self.image_paths)
    
    def preprocess_galaxy_image(self, img_np): # not used as the augmented images are preprocessed
        """Apply preprocessing to enhance galaxy features"""
        
        # 1. Bilateral filter - reduces noise while preserving edges (galaxy boundaries)
        if self.config.APPLY_DENOISING:
            img_np = cv2.bilateralFilter(img_np, d=5, sigmaColor=50, sigmaSpace=50)
        
        # 2. CLAHE (Contrast Limited Adaptive Histogram Equalization)
        # Enhances local contrast, useful for faint galaxy features
        if self.config.APPLY_CLAHE:
            lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
            l, a, b = cv2.split(lab)
            clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
            l = clahe.apply(l)
            img_np = cv2.merge([l, a, b])
            img_np = cv2.cvtColor(img_np, cv2.COLOR_LAB2RGB)
        
        # 3. Gamma correction - helps bring out faint features
        if self.config.GAMMA_CORRECTION != 1.0:
            img_np = np.power(img_np/255.0, self.config.GAMMA_CORRECTION)
            img_np = (img_np * 255).astype(np.uint8)
            
        return img_np
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Load image
        img = Image.open(img_path).convert('RGB')
        img_np = np.array(img)
        
        # Apply preprocessing
        img_np = self.preprocess_galaxy_image(img_np)
        img = Image.fromarray(img_np)
        
        # Apply augmentation (for training set)
        if self.augment and random.random() < self.config.AUGMENTATION_PROB:
            # Random horizontal flip
            if random.random() > 0.5:
                img = TF.hflip(img)
            # Random vertical flip
            if random.random() > 0.5:
                img = TF.vflip(img)
            # Random rotation for underrepresented classes
            # (In actual use, check class distribution and apply accordingly)
            if random.random() > 0.7:
                angle = random.uniform(-self.config.ROTATION_DEGREES, self.config.ROTATION_DEGREES)
                img = TF.rotate(img, angle)
        
        if self.transform:
            img = self.transform(img)
            
        return img, label


In [ ]:
class DynamicLargeKernel(nn.Module):
    """
    Dynamic Large Kernel (DLK) module from the paper.
    Adaptively selects features from two deep large kernels using sigmoid function.
    """
    
    def __init__(self, in_channels, kernel_sizes=[7, 9]):
        super().__init__()
        # Two parallel large kernel paths
        self.large_kernel1 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_sizes[0], 
                     padding=kernel_sizes[0]//2, groups=in_channels),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        self.large_kernel2 = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_sizes[1], 
                     padding=kernel_sizes[1]//2, groups=in_channels),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
        # Selection gate using sigmoid
        self.gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels, in_channels, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x):
        feat1 = self.large_kernel1(x)
        feat2 = self.large_kernel2(x)
        
        # Adaptive selection using sigmoid gate
        weight = self.gate(x)
        out = feat1 * weight + feat2 * (1 - weight)
        
        return out


class MultiScaleFeedForward(nn.Module):
    """
    Multi-Scale Feed-forward Network (MS-FFN) from the paper.
    Processes features at multiple scales for better representation.
    """
    
    def __init__(self, in_channels, expansion_ratio=4):
        super().__init__()
        hidden_channels = in_channels * expansion_ratio
        
        # Calculate channels for each branch to ensure they sum up correctly
        c1 = hidden_channels // 3
        c2 = hidden_channels // 3
        c3 = hidden_channels - c1 - c2  # Assign the remainder to the third branch
        
        # Multi-scale branches (3x3, 5x5, 7x7 as per paper concept)
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, c1, 1),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),
            nn.Conv2d(c1, c1, 3, padding=1),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True)
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, c2, 1),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=True),
            nn.Conv2d(c2, c2, 5, padding=2),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=True)
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, c3, 1),
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),
            nn.Conv2d(c3, c3, 7, padding=3),
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True)
        )
        
        # Fusion layer
        self.fusion = nn.Sequential(
            nn.Conv2d(hidden_channels, in_channels, 1),
            nn.BatchNorm2d(in_channels)
        )
        
    def forward(self, x):
        feat1 = self.branch1(x)
        feat2 = self.branch2(x)
        feat3 = self.branch3(x)
        
        # Concatenate multi-scale features
        multi_scale_feat = torch.cat([feat1, feat2, feat3], dim=1)
        
        # Fuse back to original channels
        out = self.fusion(multi_scale_feat)
        
        return out

class AttentionalFeatureFusion(nn.Module):
    """
    Attentional Feature Fusion (AFF) module from the paper.
    Better integrates features with semantic and scale inconsistencies.
    """
    
    def __init__(self, in_channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels * 2, in_channels, 1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x, residual):
        # Concatenate features
        concat_feat = torch.cat([x, residual], dim=1)
        
        # Generate attention weights
        attention_weight = self.attention(concat_feat)
        
        # Weighted fusion
        out = x * attention_weight + residual * (1 - attention_weight)
        
        return out


class DMABlock(nn.Module):
    """
    Dynamic Multiscale Attention Block - Core building block of DMAGNet
    """
    
    def __init__(self, in_channels, stride=1):
        super().__init__()
        
        # Dynamic Large Kernel
        self.dlk = DynamicLargeKernel(in_channels)
        
        # Multi-Scale FFN
        self.ms_ffn = MultiScaleFeedForward(in_channels)
        
        # Attentional Feature Fusion
        self.aff = AttentionalFeatureFusion(in_channels)
        
        # Standard convolution path (for residual)
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, 3, stride, padding=1),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True)
        )
        
        # Downsample if needed
        self.downsample = None
        if stride > 1:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, in_channels, 1, stride),
                nn.BatchNorm2d(in_channels)
            )
    
    def forward(self, x):
        residual = x
        
        # Main path: Conv -> DLK -> MS-FFN
        out = self.conv(x)
        out = self.dlk(out)
        out = self.ms_ffn(out)
        
        # Downsample residual if needed
        if self.downsample is not None:
            residual = self.downsample(residual)
        
        # Attentional feature fusion
        out = self.aff(out, residual)
        
        return out


class LayerAttentionModule(nn.Module):
    """
    Layer Attention Module (LAM) - Fuses outputs of last two DMA modules
    """
    
    def __init__(self, in_channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(in_channels * 2, in_channels, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, in_channels * 2, 1),
            nn.Sigmoid()
        )
        
    def forward(self, x1, x2):
        concat = torch.cat([x1, x2], dim=1)
        attention = self.attention(concat)
        
        # Split attention weights
        att1, att2 = torch.split(attention, x1.size(1), dim=1)
        
        # Apply attention and concatenate
        out = torch.cat([x1 * att1, x2 * att2], dim=1)
        
        return out


class AttentionPool2d(nn.Module):
    """
    Attention pooling mechanism from the paper
    """
    
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1)
        )
        
    def forward(self, x):
        return self.attention(x).flatten(1)



In [ ]:
class DMAGNet(nn.Module):
    """
    Dynamic Multiscale Attention Galaxy Network
    Architecture follows Table 3 from the paper
    """
    
    def __init__(self, num_classes, config=config):
        super().__init__()
        
        # Stem module - initial processing
        self.stem = nn.Sequential(
            nn.Conv2d(3, config.INITIAL_CHANNELS, 7, stride=2, padding=3),
            nn.BatchNorm2d(config.INITIAL_CHANNELS),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2)  # Added for better downsampling
        )
        
        # Layer 1
        self.layer1_downsample = nn.Sequential(
            nn.Conv2d(config.INITIAL_CHANNELS, config.LAYER_CHANNELS[0], 3, stride=2, padding=1),
            nn.BatchNorm2d(config.LAYER_CHANNELS[0])
        )
        self.layer1_blocks = nn.ModuleList([
            DMABlock(config.LAYER_CHANNELS[0]) for _ in range(config.NUM_DMA_BLOCKS[0])
        ])
        
        # Layer 2
        self.layer2_downsample = nn.Sequential(
            nn.Conv2d(config.LAYER_CHANNELS[0], config.LAYER_CHANNELS[1], 3, stride=2, padding=1),
            nn.BatchNorm2d(config.LAYER_CHANNELS[1])
        )
        self.layer2_blocks = nn.ModuleList([
            DMABlock(config.LAYER_CHANNELS[1]) for _ in range(config.NUM_DMA_BLOCKS[1])
        ])
        
        # Layer 3
        self.layer3_downsample = nn.Sequential(
            nn.Conv2d(config.LAYER_CHANNELS[1], config.LAYER_CHANNELS[2], 3, stride=2, padding=1),
            nn.BatchNorm2d(config.LAYER_CHANNELS[2])
        )
        self.layer3_blocks = nn.ModuleList([
            DMABlock(config.LAYER_CHANNELS[2]) for _ in range(config.NUM_DMA_BLOCKS[2])
        ])
        
        # LAM - fuses last two blocks of Layer 3
        self.lam = LayerAttentionModule(config.LAYER_CHANNELS[2])
        
        # Head module
        self.head_conv = nn.Conv2d(config.LAYER_CHANNELS[2] * 2, 256, 1)
        # CORRECTED: Adjust spatial size from 16x16 to 8x8
        self.layer_norm = nn.LayerNorm([256, 8, 8])
        self.attention_pool = AttentionPool2d(256, 256)
        self.classifier = nn.Linear(256, num_classes)
        
        # Initialize weights
        self._initialize_weights()
        
    def _initialize_weights(self):
        """Initialize model weights using He initialization"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        # Stem
        x = self.stem(x)
        
        # Layer 1
        x = self.layer1_downsample(x)
        for block in self.layer1_blocks:
            x = block(x)
        
        # Layer 2
        x = self.layer2_downsample(x)
        for block in self.layer2_blocks:
            x = block(x)
        
        # Layer 3
        x = self.layer3_downsample(x)
        layer3_features = []
        for block in self.layer3_blocks:
            x = block(x)
            layer3_features.append(x)
        
        # LAM - fuse last two Layer 3 outputs
        x = self.lam(layer3_features[-2], layer3_features[-1])
        
        # Head
        x = self.head_conv(x)
        x = self.layer_norm(x)
        x = self.attention_pool(x)
        x = self.classifier(x)
        
        return x


In [ ]:
def prepare_data_loaders(config):
    """Prepare train, validation, and test data loaders"""
    
    # Get all image paths and labels
    data_dir = Path(config.DATA_DIR)
    all_images = []
    all_labels = []
    class_names = []
    
    for class_idx, class_dir in enumerate(sorted(data_dir.iterdir())):
        if class_dir.is_dir():
            class_names.append(class_dir.name)
            for img_path in class_dir.glob("*.png"):
                all_images.append(str(img_path))
                all_labels.append(class_idx)
    
    # Convert to numpy arrays for easier splitting
    all_images = np.array(all_images)
    all_labels = np.array(all_labels)
    
    # Stratified split
    from sklearn.model_selection import train_test_split
    
    # First split: train+val vs test
    train_val_images, test_images, train_val_labels, test_labels = train_test_split(
        all_images, all_labels, 
        test_size=config.TEST_RATIO,
        stratify=all_labels,
        random_state=42
    )
    
    # Second split: train vs val
    val_size = config.VAL_RATIO / (config.TRAIN_RATIO + config.VAL_RATIO)
    train_images, val_images, train_labels, val_labels = train_test_split(
        train_val_images, train_val_labels,
        test_size=val_size,
        stratify=train_val_labels,
        random_state=42
    )
    
    # Print class distribution
    print("\nClass Distribution:")
    print(f"Classes: {class_names}")
    print(f"Total samples: {len(all_images)}")
    print(f"Train samples: {len(train_images)}")
    print(f"Val samples: {len(val_images)}")
    print(f"Test samples: {len(test_images)}")
    
    # Check class balance
    train_dist = Counter(train_labels)
    print("\nTraining set class distribution:")
    for cls_idx, count in sorted(train_dist.items()):
        print(f"  {class_names[cls_idx]}: {count} samples")
    
    # Define transforms
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                    std=[0.229, 0.224, 0.225])
    
    train_transform = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.9, 1.1)),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        normalize
    ])
    
    val_test_transform = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        normalize
    ])
    
    # Create datasets
    train_dataset = GalaxyDataset(train_images, train_labels, 
                                 transform=train_transform, augment=True)
    val_dataset = GalaxyDataset(val_images, val_labels,
                               transform=val_test_transform, augment=False)
    test_dataset = GalaxyDataset(test_images, test_labels,
                                transform=val_test_transform, augment=False)
    
    # Create data loaders with optimizations
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )
    
    return train_loader, val_loader, test_loader, class_names


In [ ]:

class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=7, min_delta=0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
        
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0


def train_epoch(model, dataloader, criterion, optimizer, scaler, device, config):
    """Train for one epoch with mixed precision training"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(dataloader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        # Mixed precision training
        if config.USE_MIXED_PRECISION:
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            
            # Gradient clipping
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP_VALUE)
            
            scaler.step(optimizer)
            scaler.update()
        else:
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), config.GRADIENT_CLIP_VALUE)
            optimizer.step()
        
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate_epoch(model, dataloader, criterion, device):
    """Validate/Test epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Validating')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
            # Store for metrics
            probs = F.softmax(outputs, dim=1)
            all_probs.append(probs.cpu().numpy())
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({
                'loss': f'{loss.item():.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(dataloader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc, np.array(all_preds), np.array(all_labels), np.vstack(all_probs)


In [ ]:

def train_model(model, train_loader, val_loader, config, num_classes, class_names):
    """Complete training loop with checkpointing and visualization"""
    
    # Create checkpoint directory
    os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, 
                          weight_decay=config.WEIGHT_DECAY)
    
    # Scheduler (Cosine Annealing as commonly used in vision models)
    scheduler = CosineAnnealingLR(optimizer, T_max=config.T_MAX, eta_min=config.MIN_LR)
    
    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if config.USE_MIXED_PRECISION else None
    
    # Early stopping
    early_stopping = EarlyStopping(patience=config.PATIENCE, min_delta=config.MIN_DELTA)
    
    # Training history
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': []
    }
    
    best_val_acc = 0.0
    
    print(f"\n{'='*60}")
    print(f"Starting Training - {config.NUM_EPOCHS} epochs")
    print(f"{'='*60}\n")
    
    for epoch in range(config.NUM_EPOCHS):
        start_time = time.time()
        
        print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
        print("-" * 40)
        
        # Training phase
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device, config
        )
        
        # Validation phase
        val_loss, val_acc, _, _, _ = validate_epoch(model, val_loader, criterion, device)
        
        # Step scheduler
        scheduler.step()
        
        # Record history
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # Print epoch results
        epoch_time = time.time() - start_time
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")
        print(f"Time: {epoch_time:.2f}s")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_acc': best_val_acc,
                'history': history,
                'config': config.__dict__
            }, f"{config.CHECKPOINT_DIR}/best_model.pth")
            print(f"✓ Saved best model (Val Acc: {best_val_acc:.2f}%)")
        
        # Regular checkpoint
        if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_acc': val_acc,
                'history': history,
                'config': config.__dict__
            }, f"{config.CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.pth")
            print(f"✓ Saved checkpoint at epoch {epoch+1}")
        
        # Early stopping check
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
            break
        
        print()
    
    print(f"\n{'='*60}")
    print(f"Training Complete! Best Val Acc: {best_val_acc:.2f}%")
    print(f"{'='*60}\n")
    
    return history


In [ ]:
def train_model_resume(model, train_loader, val_loader, config, num_classes, class_names, 
                      start_epoch=0, best_val_acc=0.0, history=None):
    """Training loop with resume capability"""
    
    # Create checkpoint directory
    os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
    
    # Initialize history if not provided
    if history is None:
        history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=config.LEARNING_RATE, 
                          weight_decay=config.WEIGHT_DECAY)
    
    # Scheduler
    scheduler = CosineAnnealingLR(optimizer, T_max=config.T_MAX, eta_min=config.MIN_LR)
    
    # If resuming, load optimizer and scheduler state
    checkpoint_path = f"{config.CHECKPOINT_DIR}/best_model.pth"
    if start_epoch > 0 and os.path.exists(checkpoint_path):
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            if 'optimizer_state_dict' in checkpoint:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if 'scheduler_state_dict' in checkpoint:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            
            # Advance scheduler to current epoch
            for _ in range(start_epoch):
                scheduler.step()
                
        except Exception as e:
            print(f"Warning: Could not restore optimizer/scheduler state: {e}")
    
    # Mixed precision scaler
    scaler = torch.cuda.amp.GradScaler() if config.USE_MIXED_PRECISION else None
    
    # Early stopping
    early_stopping = EarlyStopping(patience=config.PATIENCE, min_delta=config.MIN_DELTA)
    
    # If resuming and we have validation history, set early stopping state
    if start_epoch > 0 and len(history['val_loss']) > 0:
        # Find the best validation loss so far
        best_val_loss = min(history['val_loss'])
        early_stopping.best_loss = best_val_loss
        # Calculate how many epochs since best loss
        best_loss_epoch = history['val_loss'].index(best_val_loss)
        epochs_since_best = len(history['val_loss']) - 1 - best_loss_epoch
        early_stopping.counter = max(0, epochs_since_best)
    
    print(f"\n{'='*60}")
    print(f"{'Resuming' if start_epoch > 0 else 'Starting'} Training - Epochs {start_epoch+1} to {config.NUM_EPOCHS}")
    print(f"{'='*60}\n")
    
    for epoch in range(start_epoch, config.NUM_EPOCHS):
        start_time = time.time()
        
        print(f"Epoch {epoch+1}/{config.NUM_EPOCHS}")
        print("-" * 40)
        
        # Training phase
        train_loss, train_acc = train_epoch(
            model, train_loader, criterion, optimizer, scaler, device, config
        )
        
        # Validation phase
        val_loss, val_acc, _, _, _ = validate_epoch(model, val_loader, criterion, device)
        
        # Step scheduler
        scheduler.step()
        
        # Record history
        current_lr = optimizer.param_groups[0]['lr']
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # Print epoch results
        epoch_time = time.time() - start_time
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")
        print(f"Time: {epoch_time:.2f}s")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'best_val_acc': best_val_acc,
                'history': history,
                'config': config.__dict__
            }, checkpoint_path)
            print(f"✓ Saved best model (Val Acc: {best_val_acc:.2f}%)")
        
        # Regular checkpoint
        if (epoch + 1) % config.SAVE_EVERY_N_EPOCHS == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'val_acc': val_acc,
                'history': history,
                'config': config.__dict__
            }, f"{config.CHECKPOINT_DIR}/checkpoint_epoch_{epoch+1}.pth")
            print(f"✓ Saved checkpoint at epoch {epoch+1}")
        
        # Early stopping check
        early_stopping(val_loss)
        if early_stopping.early_stop:
            print(f"\n⚠ Early stopping triggered at epoch {epoch+1}")
            break
        
        print()
    
    print(f"\n{'='*60}")
    print(f"Training Complete! Best Val Acc: {best_val_acc:.2f}%")
    print(f"{'='*60}\n")
    
    return history

In [ ]:

def evaluate_model(model, test_loader, class_names, device):
    """Comprehensive model evaluation with metrics and visualizations"""
    
    print("\n" + "="*60)
    print("MODEL EVALUATION ON TEST SET")
    print("="*60 + "\n")
    
    # Get predictions
    criterion = nn.CrossEntropyLoss()
    test_loss, test_acc, all_preds, all_labels, all_probs = validate_epoch(
        model, test_loader, criterion, device
    )
    
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    
    # Detailed classification report
    print("\nClassification Report:")
    print("-" * 40)
    print(classification_report(all_labels, all_preds, 
                               target_names=class_names, digits=4))
    
    # Confusion Matrix
    cm = confusion_matrix(all_labels, all_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix.png', dpi=150)
    plt.show()
    
    # ROC Curves and AUC
    num_classes = len(class_names)
    all_labels_bin = label_binarize(all_labels, classes=range(num_classes))
    
    plt.figure(figsize=(12, 8))
    for i in range(num_classes):
        fpr, tpr, _ = roc_curve(all_labels_bin[:, i], all_probs[:, i])
        auc_score = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f'{class_names[i]} (AUC = {auc_score:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves for Galaxy Classification')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('roc_curves.png', dpi=150)
    plt.show()
    
    # Per-class accuracy
    print("\nPer-Class Accuracy:")
    print("-" * 40)
    for i, class_name in enumerate(class_names):
        class_mask = all_labels == i
        class_acc = (all_preds[class_mask] == all_labels[class_mask]).mean() * 100
        print(f"{class_name:20s}: {class_acc:.2f}%")
    
    return test_loss, test_acc, cm


In [ ]:
def plot_training_history(history):
    """Plot training history"""
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # Loss plot
    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Accuracy plot
    axes[1].plot(history['train_acc'], label='Train Acc', linewidth=2)
    axes[1].plot(history['val_acc'], label='Val Acc', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    # Learning rate plot
    axes[2].plot(history['lr'], linewidth=2, color='green')
    axes[2].set_xlabel('Epoch')
    axes[2].set_ylabel('Learning Rate')
    axes[2].set_title('Learning Rate Schedule')
    axes[2].grid(alpha=0.3)
    axes[2].set_yscale('log')
    
    plt.tight_layout()
    plt.savefig('training_history.png', dpi=150)
    plt.show()


In [ ]:
def main():
    """Main training pipeline with automatic checkpoint resuming"""
    
    print("="*60)
    print("DMAGNet GALAXY MORPHOLOGICAL CLASSIFICATION")
    print("="*60)
    
    # Load data
    print("\nLoading and preparing data...")
    train_loader, val_loader, test_loader, class_names = prepare_data_loaders(config)
    num_classes = len(class_names)
    
    # Initialize model
    print("\nInitializing DMAGNet model...")
    model = DMAGNet(num_classes=num_classes, config=config).to(device)
    
    # Check for existing checkpoint and auto-resume
    checkpoint_path = f"{config.CHECKPOINT_DIR}/best_model.pth"
    start_epoch = 0
    best_val_acc = 0.0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    
    if os.path.exists(checkpoint_path):
        print(f"\nFound existing checkpoint at {checkpoint_path}")
        print("Resuming training from checkpoint...")
        try:
            checkpoint = torch.load(checkpoint_path, map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_val_acc = checkpoint['best_val_acc']
            
            # Restore training history if available
            if 'history' in checkpoint:
                history = checkpoint['history']
            
            print(f"Resumed from epoch {checkpoint['epoch']} with Val Acc: {best_val_acc:.2f}%")
            print(f"Will continue training from epoch {start_epoch}")
            
            # If we've already completed all epochs, just evaluate
            if start_epoch >= config.NUM_EPOCHS:
                print(f"Training already completed ({config.NUM_EPOCHS} epochs). Proceeding to evaluation...")
                # Skip training, go directly to evaluation
                plot_training_history(history)
                test_loss, test_acc, conf_matrix = evaluate_model(model, test_loader, class_names, device)
                
                # Save final results
                results = {
                    'test_accuracy': test_acc,
                    'test_loss': test_loss,
                    'training_history': history,
                    'confusion_matrix': conf_matrix.tolist(),
                    'class_names': class_names,
                    'config': config.__dict__
                }
                
                with open('training_results.json', 'w') as f:
                    json.dump(results, f, indent=4, default=str)
                
                print("\n" + "="*60)
                print("EVALUATION COMPLETE!")
                print(f"Final Test Accuracy: {test_acc:.2f}%")
                print("Results saved to training_results.json")
                print("="*60)
                return
                
        except Exception as e:
            print(f"Error loading checkpoint: {e}")
            print("Starting fresh training...")
            start_epoch = 0
            best_val_acc = 0.0
            history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [], 'lr': []}
    else:
        print("\nNo existing checkpoint found. Starting fresh training...")
    
    # Train model (with modified function to accept start_epoch and history)
    history = train_model_resume(model, train_loader, val_loader, config, num_classes, 
                                class_names, start_epoch, best_val_acc, history)
    
    # Plot training history
    plot_training_history(history)
    
    # Load best model for evaluation
    print("\nLoading best model for evaluation...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Evaluate on test set
    test_loss, test_acc, conf_matrix = evaluate_model(model, test_loader, class_names, device)
    
    # Save final results
    results = {
        'test_accuracy': test_acc,
        'test_loss': test_loss,
        'training_history': history,
        'confusion_matrix': conf_matrix.tolist(),
        'class_names': class_names,
        'config': config.__dict__
    }
    
    with open('training_results.json', 'w') as f:
        json.dump(results, f, indent=4, default=str)
    
    print("\n" + "="*60)
    print("TRAINING COMPLETE!")
    print(f"Final Test Accuracy: {test_acc:.2f}%")
    print("Results saved to training_results.json")
    print("="*60)


if __name__ == "__main__":
    main()

In [ ]:
def reorganize_classes_4class_benchmark(model, test_loader, original_class_names, device):
    """
    Reorganize classes by:
    1. Keeping 'Disturbed' class
    2. Merging 3 spiral classes into 'Spiral'
    3. Merging 2 smooth classes into 'Smooth'
    4. Keeping 'Merging' as is
    Final classes: ['Disturbed', 'Merging', 'Smooth', 'Spiral']
    """
    
    print("\n" + "="*70)
    print("4-CLASS REORGANIZATION AND NEW BENCHMARKING")
    print("="*70 + "\n")
    
    # Define class mapping
    print("Original classes:", original_class_names)
    
    # Create mapping from original classes to new classes
    class_mapping = {}
    new_class_names = ['Disturbed', 'Merging', 'Smooth', 'Spiral']
    
    for i, class_name in enumerate(original_class_names):
        if 'Disturbed' in class_name:
            class_mapping[i] = 0  # Map to 'Disturbed'
        elif 'Merging' in class_name:
            class_mapping[i] = 1  # Map to 'Merging'
        elif any(smooth in class_name for smooth in ['Round Smooth', 'In-Between Round Smooth']):
            class_mapping[i] = 2  # Map to 'Smooth'
        elif any(spiral in class_name for spiral in ['Barred Spiral', 'Unbarred Loose Spiral', 'Unbarred Tight Spiral']):
            class_mapping[i] = 3  # Map to 'Spiral'
        else:
            class_mapping[i] = None  # Remove any other unknown classes
    
    print(f"\nClass mapping:")
    for orig_idx, orig_name in enumerate(original_class_names):
        new_idx = class_mapping[orig_idx]
        if new_idx is not None:
            print(f"  {orig_name} -> {new_class_names[new_idx]}")
        else:
            print(f"  {orig_name} -> [REMOVED]")
    
    print(f"\nNew 4 classes: {new_class_names}")
    
    # Get predictions on test set
    model.eval()
    all_original_preds = []
    all_original_labels = []
    all_original_probs = []
    
    print("\nGetting predictions on test set...")
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc='Predicting'):
            images = images.to(device)
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            
            all_original_preds.extend(predicted.cpu().numpy())
            all_original_labels.extend(labels.numpy())
            all_original_probs.append(probs.cpu().numpy())
    
    all_original_probs = np.vstack(all_original_probs)
    
    # Reorganize predictions and labels
    new_labels = []
    new_preds = []
    new_probs = []
    
    for i in range(len(all_original_labels)):
        orig_label = all_original_labels[i]
        orig_pred = all_original_preds[i]
        orig_prob = all_original_probs[i]
        
        # Map original label to new label
        new_label = class_mapping[orig_label]
        
        # For predictions, we need to aggregate probabilities for merged classes
        new_prob = np.zeros(len(new_class_names))
        for orig_class_idx, new_class_idx in class_mapping.items():
            if new_class_idx is not None:
                new_prob[new_class_idx] += orig_prob[orig_class_idx]
        
        new_pred = np.argmax(new_prob)
        
        new_labels.append(new_label)
        new_preds.append(new_pred)
        new_probs.append(new_prob)
    
    new_labels = np.array(new_labels)
    new_preds = np.array(new_preds)
    new_probs = np.array(new_probs)
    
    new_accuracy = (new_preds == new_labels).mean() * 100
    print(f"\n4-Class Test Accuracy: {new_accuracy:.2f}%")
    
    # Detailed classification report
    print("\nClassification Report (4 Classes):")
    print("-" * 50)
    print(classification_report(new_labels, new_preds, 
                               target_names=new_class_names, digits=4))
    
    # Class distribution
    print("\nClass Distribution in Test Set:")
    print("-" * 50)
    unique_labels, counts = np.unique(new_labels, return_counts=True)
    for label, count in zip(unique_labels, counts):
        percentage = count / len(new_labels) * 100
        print(f"  {new_class_names[label]:15s}: {count:4d} samples ({percentage:5.1f}%)")
    
    # Confusion Matrix for new classes
    new_cm = confusion_matrix(new_labels, new_preds)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(new_cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=new_class_names, yticklabels=new_class_names)
    plt.title('Confusion Matrix - 4-Class Reorganization')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig('confusion_matrix_4class.png', dpi=150)
    plt.show()
    
    # ROC Curves for new classes
    plt.figure(figsize=(12, 8))
    colors = ['red', 'blue', 'green', 'orange']
    for i in range(len(new_class_names)):
        # Binarize labels for this class
        binary_labels = (new_labels == i).astype(int)
        class_probs = new_probs[:, i]
        
        fpr, tpr, _ = roc_curve(binary_labels, class_probs)
        auc_score = auc(fpr, tpr)
        plt.plot(fpr, tpr, linewidth=2, color=colors[i],
                label=f'{new_class_names[i]} (AUC = {auc_score:.3f})')
    
    plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves - 4-Class Reorganization')
    plt.legend(loc='lower right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('roc_curves_4class.png', dpi=150)
    plt.show()
    
    # Per-class metrics
    print("\nPer-Class Metrics:")
    print("-" * 50)
    for i, class_name in enumerate(new_class_names):
        # Precision, Recall, F1-Score
        class_mask = new_labels == i
        if class_mask.sum() > 0:
            class_acc = (new_preds[class_mask] == new_labels[class_mask]).mean() * 100
            
            # True positives, false positives, false negatives
            tp = ((new_preds == i) & (new_labels == i)).sum()
            fp = ((new_preds == i) & (new_labels != i)).sum()
            fn = ((new_preds != i) & (new_labels == i)).sum()
            
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0
            recall = tp / (tp + fn) if (tp + fn) > 0 else 0
            f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
            
            print(f"{class_name:15s}: Acc={class_acc:5.1f}%, "
                  f"Prec={precision:.3f}, Rec={recall:.3f}, F1={f1:.3f}")
    
    reorganized_4class_results = {
        'new_class_names': new_class_names,
        'class_mapping': {original_class_names[k]: new_class_names[v] 
                         for k, v in class_mapping.items() if v is not None},
        'new_accuracy': new_accuracy,
        'confusion_matrix': new_cm.tolist(),
        'class_distribution': {new_class_names[label]: int(count) 
                              for label, count in zip(unique_labels, counts)},
        'samples_total': len(new_labels),
        'num_classes': len(new_class_names)
    }
    
    # Save reorganized results
    with open('reorganized_4class_results.json', 'w') as f:
        json.dump(reorganized_4class_results, f, indent=4)
    
    print(f"\nResults saved to: reorganized_4class_results.json")
    print(f"Visualizations saved: confusion_matrix_4class.png, roc_curves_4class.png")
    
    return new_accuracy, new_cm, reorganized_4class_results


def run_complete_4class_evaluation():
    """
    Complete pipeline to run 4-class reorganized evaluation
    """
    
    # Load the trained model
    checkpoint_path = f"{config.CHECKPOINT_DIR}/best_model.pth"
    
    if not os.path.exists(checkpoint_path):
        print(f"Error: No trained model found at {checkpoint_path}")
        print("Please run the training pipeline first.")
        return
    
    print("Loading trained model for 4-class evaluation...")
    
    # Load data to get class names
    train_loader, val_loader, test_loader, original_class_names = prepare_data_loaders(config)
    num_classes = len(original_class_names)
    
    # Load model
    model = DMAGNet(num_classes=num_classes, config=config).to(device)
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    
    # Run 4-class reorganized evaluation
    new_accuracy, new_cm, results = reorganize_classes_4class_benchmark(
        model, test_loader, original_class_names, device
    )
    
    print("\n" + "="*70)
    print("4-CLASS REORGANIZED EVALUATION COMPLETE!")
    print(f"New 4-Class Accuracy: {new_accuracy:.2f}%")
    print("="*70)
    
    return results


def run_comparison_analysis():
    """
    Run comparison between 3-class and 4-class reorganizations
    """
    
    print("\n" + "="*80)
    print("COMPREHENSIVE RESULTS: 4-CLASS CLASSIFICATION")
    print("="*80 + "\n")
    
    # Run 4-class evaluation
    results_4class = run_complete_4class_evaluation()
    
    return results_4class

# Run the 4-class reorganized evaluation
if __name__ == "__main__":
    # This can be run independently if model is already trained
    results_4class = run_comparison_analysis()

In [ ]:
# SINGLE INFERENCE

KERNEL_SIZE = 5
GAMMA = 1.6

# These are functions that were used to preprocess the images before training too

def remove_salt_pepper_noise(img_np, kernel_size=3):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened = cv2.morphologyEx(img_np, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return closed

def preprocess_single_image(random_img_path):
    
    original_img = Image.open(random_img_path).convert('RGB')
    original_np = np.array(original_img)
    
    img_np = original_np.copy()
    
    img_morph = remove_salt_pepper_noise(img_np, kernel_size=KERNEL_SIZE)
    img_median = cv2.medianBlur(img_np, ksize=KERNEL_SIZE)
    img_bilateral = img_median.copy()
    img_bilateral = cv2.bilateralFilter(img_bilateral, d=5, sigmaColor=50, sigmaSpace=50)
    
    img_clahe = img_bilateral.copy()
    lab = cv2.cvtColor(img_clahe, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    img_clahe = cv2.merge([l, a, b])
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2RGB)
    img_gamma = img_clahe.copy()

    img_gamma = np.power(img_gamma/255.0, GAMMA)
    img_gamma = (img_gamma * 255).astype(np.uint8)

    img = Image.fromarray(img_gamma)
    
    # Apply transforms
    transform = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    img_tensor = transform(img).unsqueeze(0)
    return img_tensor, img_gamma


def predict_galaxy_morphology(image_path, model, device, config):
    """
    Predict galaxy morphology for a single image using 4-class classification
    """
    # Define class mappings (7-class to 4-class)
    class_mapping = {
        0: 0,  # Barred Spiral -> Spiral
        1: 3,  # Disturbed -> Disturbed
        2: 1,  # In-between Round Smooth -> Smooth
        3: 2,  # Merging -> Merging
        4: 1,  # Round Smooth -> Smooth
        5: 0,  # Unbarred Loose Spiral -> Spiral
        6: 0   # Unbarred Tight Spiral -> Spiral
    }
    
    # 4-class names
    class_names_4 = [
        'Spiral Galaxies',
        'Smooth Galaxies', 
        'Merging Galaxies',
        'Disturbed Galaxies'
    ]

    class_names_7 = [
        'Barred Spiral Galaxies',
        'Disturbed Galaxies',
        'In-between Round Smooth Galaxies',
        'Merging Galaxies',
        'Round Smooth Galaxies',
        'Unbarred Loose Spiral Galaxies',
        'Unbarred Tight Spiral Galaxies'
    ]
    
    # Preprocess image
    img_tensor, processed_img = preprocess_single_image(image_path)
    img_tensor = img_tensor.to(device)
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probs_7class = F.softmax(outputs, dim=1).cpu().numpy()[0]
    
    # Convert 7-class probabilities to 4-class
    probs_4class = np.zeros(4)
    for orig_idx, new_idx in class_mapping.items():
        probs_4class[new_idx] += probs_7class[orig_idx]
    
    # Get predicted class
    predicted_class = np.argmax(probs_4class)
    confidence = probs_4class[predicted_class]

    return predicted_class, confidence, probs_4class, class_names_4, processed_img, probs_7class, class_names_7

def display_prediction_results(image_path, predicted_class, confidence, probs_4class, class_names_4, processed_img, probs_7class, class_names_7):
    """
    Display prediction results with visualization
    """
    print("="*70)
    print("GALAXY MORPHOLOGY PREDICTION RESULTS")
    print("="*70)
    print(f"Image: {os.path.basename(image_path)}")
    print(f"Predicted Class: {class_names_4[predicted_class]}")
    print(f"Predicted Class: {class_names_7[np.argmax(probs_7class)]} (subclassification)") # Does subclassification
    print(f"Confidence: {confidence:.3f} ({confidence*100:.1f}%)")
    print()
    
    print("Confidence Scores for All Classes:")
    print("-" * 40)
    for i, (class_name, prob) in enumerate(zip(class_names_4, probs_4class)):
        indicator = "★" if i == predicted_class else " "
        print(f"{indicator} {class_name:25s}: {prob:.3f} ({prob*100:.1f}%)")
    print()
    
    # Visualization
    fig, ax1 = plt.subplots(1, 1, figsize=(15, 6))

    # Original image
    ax1.imshow(processed_img)
    ax1.set_title(f'Galaxy Image\n{os.path.basename(image_path)}', fontsize=12)

    plt.tight_layout()
    plt.show()

# ============================================================================
# INFERENCE EXAMPLE
# ============================================================================

# Example usage - replace with your image path
image_path = "../FinalTest/galaxy10Processed/Merging Galaxies/image_01912.png"  # Change this to your image path

# Check if image exists
if os.path.exists(image_path):
    print(f"Processing image: {image_path}")
    
    # Make prediction
    predicted_class, confidence, probs_4class, class_names_4, processed_img, probs_7class, class_names_7 = predict_galaxy_morphology(
        image_path, model, device, config
    )
    
    # Display results
    display_prediction_results(
        image_path, predicted_class, confidence, probs_4class, class_names_4, processed_img, probs_7class, class_names_7
    )
    
else:
    print(f"Image not found: {image_path}")
    print("Please update the image_path variable with a valid image path.")



In [ ]:
# This cell is used to process images for Part 3, correlation

KERNEL_SIZE = 5
GAMMA = 1.6

def remove_salt_pepper_noise(img_np, kernel_size=3):
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    opened = cv2.morphologyEx(img_np, cv2.MORPH_OPEN, kernel)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel)
    return closed

def preprocess_single_image_np(original_np):
    
    img_np = original_np.copy()
    
    img_morph = remove_salt_pepper_noise(img_np, kernel_size=KERNEL_SIZE)
    img_median = cv2.medianBlur(img_np, ksize=KERNEL_SIZE)
    img_bilateral = img_median.copy()
    img_bilateral = cv2.bilateralFilter(img_bilateral, d=5, sigmaColor=50, sigmaSpace=50)
    
    img_clahe = img_bilateral.copy()
    lab = cv2.cvtColor(img_clahe, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    img_clahe = cv2.merge([l, a, b])
    img_clahe = cv2.cvtColor(img_clahe, cv2.COLOR_LAB2RGB)
    img_gamma = img_clahe.copy()

    img_gamma = np.power(img_gamma/255.0, GAMMA)
    img_gamma = (img_gamma * 255).astype(np.uint8)

    img = Image.fromarray(img_gamma)
    
    # Apply transforms
    transform = transforms.Compose([
        transforms.Resize((config.IMG_SIZE, config.IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                           std=[0.229, 0.224, 0.225])
    ])
    
    img_tensor = transform(img).unsqueeze(0)
    return img_tensor, img_gamma


def predict_galaxy_morphology_np(original_np, model, device, config):
    """
    Predict galaxy morphology for a single image using 4-class classification
    """
    # Define class mappings (7-class to 4-class)
    class_mapping = {
        0: 0,  # Barred Spiral -> Spiral
        1: 3,  # Disturbed -> Disturbed
        2: 1,  # In-between Round Smooth -> Smooth
        3: 2,  # Merging -> Merging
        4: 1,  # Round Smooth -> Smooth
        5: 0,  # Unbarred Loose Spiral -> Spiral
        6: 0   # Unbarred Tight Spiral -> Spiral
    }
    
    # 4-class names
    class_names_4 = [
        'Spiral Galaxies',
        'Smooth Galaxies', 
        'Merging Galaxies',
        'Disturbed Galaxies'
    ]
    
    # Preprocess image
    img_tensor, processed_img = preprocess_single_image_np(original_np=original_np)
    img_tensor = img_tensor.to(device)
    
    # Make prediction
    model.eval()
    with torch.no_grad():
        outputs = model(img_tensor)
        probs_7class = F.softmax(outputs, dim=1).cpu().numpy()[0]
    
    # Convert 7-class probabilities to 4-class
    probs_4class = np.zeros(4)
    for orig_idx, new_idx in class_mapping.items():
        probs_4class[new_idx] += probs_7class[orig_idx]
    
    # Get predicted class
    predicted_class = np.argmax(probs_4class)
    confidence = probs_4class[predicted_class]
    
    return predicted_class


base_dir = Path("../Part3/processed_galaxy_data/")
subdirs = [d for d in base_dir.iterdir() if d.is_dir()]

for folder in subdirs:
    images = np.load(folder / "images.npy")
    labels = [predict_galaxy_morphology_np(image, model, device, config) for image in images]
    data_df = pd.read_csv(folder / "data.csv")
    data_df['predicted_class'] = labels
    data_df.to_csv(folder / "new_data.csv", index=False)
    print(f"Processed folder: {folder}")
